In [1]:
!pip install -q gradio

In [5]:
import requests
import gradio as gr
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [10]:
df = pd.read_csv('df_mejora_2.csv')
df.head()

,provincia,dia_semana,hospital,temperatura_media,temperatura_maxima,temperatura_minima,viento,viento_maximo,precipitaciones,humedad,mes,year,pacientes_ayer,target_pacientes,fecha_hora
0,2,0,2,3.7,11.0,-3.5,5.9,14.8,0.00,78.0,1,2024,645,1,2024-01-01 00:07:00
1,5,0,3,4.6,9.7,0.9,5.7,9.4,0.51,86.0,1,2024,645,2,2024-01-01 00:10:00
2,1,0,1,3.9,8.0,0.0,11.3,24.1,0.00,79.0,1,2024,645,3,2024-01-01 00:12:00
3,1,0,1,3.9,8.0,0.0,11.3,24.1,0.00,79.0,1,2024,645,4,2024-01-01 00:15:00
4,2,0,2,3.7,11.0,-3.5,5.9,14.8,0.00,78.0,1,2024,645,5,2024-01-01 00:16:00


In [6]:
coords = {
    "Ávila": (40.6565, -4.6818),
    "Burgos": (42.3439, -3.6969),
    "León": (42.5987, -5.5671),
    "Salamanca": (40.9701, -5.6635),
    "Segovia": (40.9429, -4.1088),
    "Soria": (41.7636, -2.4649),
    "Valladolid": (41.6523, -4.7245),
    "Zamora": (41.5033, -5.7446)
}

,provincia,temperatura_media,temperatura_maxima,temperatura_minima,viento_medio,viento_maximo,humedad_media,precipitaciones,fecha_mañana
0,Burgos,18.48,26.2,12.6,17.23,26.8,63.92,0.0,2026-06-08


In [3]:
def get_data(coords, provincia):
  # Obtener coordenadas automáticamente
  lat, lon = coords[provincia]

  # URL Open-Meteo
  url = (
      "https://api.open-meteo.com/v1/forecast"
      f"?latitude={lat}"
      f"&longitude={lon}"
      "&hourly="
      "temperature_2m,"
      "relative_humidity_2m,"
      "wind_speed_10m,"
      "precipitation"
      "&forecast_days=2"
  )

  # Petición API
  response = requests.get(url)

  # JSON
  data = response.json()

  # DataFrame
  df_weather = pd.DataFrame({
      "fecha_hora": data["hourly"]["time"],
      "temperatura": data["hourly"]["temperature_2m"],
      "humedad": data["hourly"]["relative_humidity_2m"],
      "viento": data["hourly"]["wind_speed_10m"],
      "precipitacion": data["hourly"]["precipitation"]
  })

  # Convertir fecha
  df_weather["fecha_hora"] = pd.to_datetime(df_weather["fecha_hora"])

  # Obtener mañana
  mañana = (datetime.now() + timedelta(days=1)).date()

  # Filtrar mañana
  df_mañana = df_weather[
      df_weather["fecha_hora"].dt.date == mañana
  ]

  # Resumen meteorológico
  resultado = {
      "provincia": provincia,
      "temperatura_media": round(df_mañana["temperatura"].mean(), 2),
      "temperatura_maxima": df_mañana["temperatura"].max(),
      "temperatura_minima": df_mañana["temperatura"].min(),
      "viento_medio": round(df_mañana["viento"].mean(), 2),
      "viento_maximo": df_mañana["viento"].max(),
      "humedad_media": round(df_mañana["humedad"].mean(), 2),
      "precipitaciones": round(df_mañana["precipitacion"].sum(), 2),
      "fecha_mañana": mañana
  }

  return pd.DataFrame({k: [v] for k, v in resultado.items()})

In [13]:
df_sarimax = df.copy()
df_sarimax = df_sarimax.rename(columns={'fecha_hora': 'ds', 'target_pacientes': 'y'})
df_sarimax['ds'] = pd.to_datetime(df_sarimax['ds'])
df_sarimax = df_sarimax.sort_values('ds').reset_index(drop=True)

In [14]:
fecha_corte = df_sarimax['ds'].max() - pd.Timedelta(days=30)
train = df_sarimax[df_sarimax['ds'] <= fecha_corte].copy()
test = df_sarimax[df_sarimax['ds'] > fecha_corte].copy()

In [15]:
columnas_prediccion = [
    "hospital", "provincia", "dia_semana", "mes", "year", "pacientes_ayer",
    "temperatura_media", "temperatura_maxima", "temperatura_minima",
    "viento", "viento_maximo", "humedad", "precipitaciones"
]

y_train = train['y']
X_train = train[columnas_prediccion]

X_test = test[columnas_prediccion]

In [16]:
model_sarimax = sm.tsa.statespace.SARIMAX(
    endog=y_train,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 24)
)

In [ ]:
resultado_sarimax = model_sarimax.fit(disp=False)

In [ ]:
prediccion = resultado_sarimax.get_forecast(steps=len(test), exog=X_test)